In [58]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")
print(type(model))
print("Device:", model.device)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 321.99it/s]


Model loaded successfully!
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Device: cpu


In [59]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: datasets in c:\users\taher\appdata\local\programs\python\python310\lib\site-packages (5.0.1)




[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
from datasets import load_dataset
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [61]:
dataset  = load_dataset("FinGPT/fingpt-sentiment-train")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 76772
    })
})


In [62]:
example = dataset["train"][0]
print(example)

{'input': 'Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .', 'output': 'neutral', 'instruction': 'What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.'}


In [63]:
print("INSTRUCTION:")
print(example["instruction"])

print("\nINPUT:")
print(example["input"])

print("\nOUTPUT:")
print(example["output"])

INSTRUCTION:
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

INPUT:
Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .

OUTPUT:
neutral


In [64]:
messages = [
    {
        "role": "system",
        "content": "You are a financial sentiment classifier."
    },
    {
        "role": "user",
        "content": example["instruction"] + "\n\n" + example["input"]
    },
    {
        "role": "assistant",
        "content": example["output"]
    }
]

print(messages)

[{'role': 'system', 'content': 'You are a financial sentiment classifier.'}, {'role': 'user', 'content': 'What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.\n\nTeollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .'}, {'role': 'assistant', 'content': 'neutral'}]


In [65]:
from transformers import AutoTokenizer

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<|im_start|>system
You are a financial sentiment classifier.<|im_end|>
<|im_start|>user
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.

Teollisuuden Voima Oyj , the Finnish utility known as TVO , said it shortlisted Mitsubishi Heavy s EU-APWR model along with reactors from Areva , Toshiba Corp. , GE Hitachi Nuclear Energy and Korea Hydro & Nuclear Power Co. .<|im_end|>
<|im_start|>assistant
neutral<|im_end|>



In [66]:
tokenized = tokenizer(
    formatted_text,
    return_tensors="pt"
)

print(tokenized["input_ids"].shape)

torch.Size([1, 99])


In [67]:
print("Number of tokens:", tokenized["input_ids"].shape[1])

Number of tokens: 99


In [68]:
print(tokenized["input_ids"][0])

tensor([151644,   8948,    198,   2610,    525,    264,   5896,  25975,  33365,
            13, 151645,    198, 151644,    872,    198,   3838,    374,    279,
         25975,    315,    419,   3669,     30,   5209,   5157,    458,   4226,
           504,    314,  42224,     14,  59568,     14,  30487,     92,    382,
          6639,    965,  62748,  60127,  28079,   7523,    506,     88,     73,
          1154,    279,  57853,  15549,   3881,    438,   5883,     46,   1154,
          1053,    432,   2805,  31240,  78553,  28101,    274,   9812,     12,
          2537,  17925,   1614,   3156,    448,  70473,    504,   8713,   6586,
          1154,  95441,  21863,     13,   1154,  29857,  15882,  30364,  37444,
         12354,    323,  11862,  39502,    609,  37444,   7420,   3539,     13,
           659, 151645,    198, 151644,  77091,    198,  59568, 151645,    198])


In [69]:
tokens = tokenizer.convert_ids_to_tokens(
    tokenized["input_ids"][0]
)

print(tokens)

['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġa', 'Ġfinancial', 'Ġsentiment', 'Ġclassifier', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'What', 'Ġis', 'Ġthe', 'Ġsentiment', 'Ġof', 'Ġthis', 'Ġnews', '?', 'ĠPlease', 'Ġchoose', 'Ġan', 'Ġanswer', 'Ġfrom', 'Ġ{', 'negative', '/', 'neutral', '/', 'positive', '}', '.ĊĊ', 'Te', 'oll', 'isu', 'uden', 'ĠVo', 'ima', 'ĠO', 'y', 'j', 'Ġ,', 'Ġthe', 'ĠFinnish', 'Ġutility', 'Ġknown', 'Ġas', 'ĠTV', 'O', 'Ġ,', 'Ġsaid', 'Ġit', 'Ġshort', 'listed', 'ĠMitsubishi', 'ĠHeavy', 'Ġs', 'ĠEU', '-', 'AP', 'WR', 'Ġmodel', 'Ġalong', 'Ġwith', 'Ġreactors', 'Ġfrom', 'ĠAre', 'va', 'Ġ,', 'ĠToshiba', 'ĠCorp', '.', 'Ġ,', 'ĠGE', 'ĠHit', 'achi', 'ĠNuclear', 'ĠEnergy', 'Ġand', 'ĠKorea', 'ĠHydro', 'Ġ&', 'ĠNuclear', 'ĠPower', 'ĠCo', '.', 'Ġ.', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ', 'neutral', '<|im_end|>', 'Ċ']


In [70]:
%pip install -U peft

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(lora_config)



LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [72]:
lora_model = get_peft_model(
    model,
    lora_config
)

lora_model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [73]:
lora_model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [74]:
from datasets import load_dataset

dataset = load_dataset(
    "FinGPT/fingpt-sentiment-train",
    split="train"
)

small_dataset = dataset.select(range(500))

print(small_dataset)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 500
})
